# Save the Tasmanian Devil Disease & Genetic Isolation Tracker

## 1. Project Vision & Context
The Tasmanian Devil is heavily threatened by two distinct strains of transmissible cancer: Devil Facial Tumor Disease (DFT1) and DFT2. To prevent extinction, conservation programs manage a "metapopulation"—a delicate network of wild populations, isolated island reserves (like Maria Island), and mainland insurance sanctuaries.

This project builds a professional, production-grade MySQL relational database system designed to solve the two biggest operational crises conservationists face: Inbreeding Depression in captive breeding programs, and Biosecurity Breaches where infected animals risk contaminating clean sanctuary zones.

## 2. Real-World Scientific Boundaries (Our Data Rules)
To ensure the project looks authentic, the data layer enforces strict biological and geographical rules:
- **The Disease Strains**:
  - DFT1: Widespread across Tasmania since 1996.
  - DFT2: Discovered in 2014, geographically restricted mostly to the D'Entrecasteaux / Channel Peninsula in the southeast.
- **Life Expectancy Constraint**: Devils diagnosed with active DFTD symptoms rarely survive past 6 to 12 months.
- **Pedigree Structure**: Captive and sanctuary populations must track parental lineages strictly (mother_id and father_id) to monitor genetic diversity and calculate kinship coefficients.

### Step 1: Environment Configuration
We load database credentials from the local `.env` file. 
Make sure to update your MySQL credentials in `.env` if you encounter connection errors.

In [ ]:
import os
from dotenv import load_dotenv

# Force override existing loaded environment variables to prevent caching
dotenv_path = os.path.join(os.getcwd(), ".env")
load_dotenv(dotenv_path, override=True)
print("Database Configuration:")
print(f"Host: {os.getenv('DB_HOST')}")
print(f"Port: {os.getenv('DB_PORT')}")
print(f"User: {os.getenv('DB_USER')}")
print(f"Database: {os.getenv('DB_NAME')}")

### Step 2: Initialize Database Schema
We read `schema.sql` and run it on the MySQL server to establish the table layout and reset the database.

In [ ]:
import mysql.connector
import importlib
import data_factory
importlib.reload(data_factory)
from data_factory import init_schema

try:
    init_schema()
    print("\nSuccess: Schema and tables created.")
except Exception as e:
    print(f"\nError: {e}")
    print("Please check your MySQL configuration in the .env file.")

### Step 3: Populate Mock Data
We execute the demographic data factory to insert our 10 sanctuaries, 3 disease strains, ~9,000 devils, and ~40,000 health logs (enforcing all real-world scientific constraints).

In [ ]:
import importlib
import data_factory
importlib.reload(data_factory)
from data_factory import get_db_connection, populate_static_data, generate_demographics

try:
    conn = get_db_connection()
    populate_static_data(conn)
    generate_demographics()
    conn.close()
    print("\nSuccess: Database fully populated with ~50,000 realistic rows!")
except Exception as e:
    print(f"\nError populating database: {e}")

### Step 4: Test Recursive Kinship CTE Query
We query for overlapping ancestors within 3 generations for two sample devils.

In [ ]:
import pandas as pd
import importlib
import data_factory
importlib.reload(data_factory)
from data_factory import get_db_connection

conn = get_db_connection()
cursor = conn.cursor()

# Fetch a living female and living male of breeding age to evaluate
cursor.execute("SELECT d1.devil_id, d2.devil_id FROM devils d1 JOIN devils d2 ON d1.mother_id=d2.mother_id WHERE d1.sex='F' AND d2.sex='M' AND d1.mother_id IS NOT NULL AND d1.status='Alive' AND d2.status='Alive' LIMIT 1")
row = cursor.fetchone()
cursor.close()

if row:
    f_id, m_id = row
    print(f"Evaluating compatibility for related pair: Female #{f_id} and Male #{m_id}\n")
    
    q_kinship = """
        WITH RECURSIVE Ancestry AS (
            SELECT devil_id, mother_id, father_id, name, 0 AS generation, devil_id AS lineage_start
            FROM devils WHERE devil_id IN (%s, %s)
            UNION ALL
            SELECT d.devil_id, d.mother_id, d.father_id, d.name, a.generation + 1 AS generation, a.lineage_start
            FROM devils d
            INNER JOIN Ancestry a ON d.devil_id = a.mother_id OR d.devil_id = a.father_id
            WHERE a.generation < 3
        )
        SELECT 
            a.devil_id AS shared_ancestor_id,
            a.name AS shared_ancestor_name,
            a.generation AS female_side_gen,
            b.generation AS male_side_gen,
            (a.generation + b.generation) AS degree_of_relationship,
            POWER(0.5, a.generation + b.generation + 1) AS kinship_contribution
        FROM Ancestry a
        INNER JOIN Ancestry b ON a.devil_id = b.devil_id
        WHERE a.lineage_start = %s AND b.lineage_start = %s
    """
    df = pd.read_sql(q_kinship, conn, params=(f_id, m_id, f_id, m_id))
    if df.empty:
        print("Approved: No shared ancestors found up to 3 generations.")
    else:
        print("Warning: Shared ancestors found:")
        print(df)
else:
    print("Please populate the database first.")
conn.close()

### Step 5: Test Sanctuary Biosecurity Trigger
We assert that attempting to transfer an infected devil to a clean sanctuary is blocked by our database trigger.

In [ ]:
import importlib
import data_factory
importlib.reload(data_factory)
from data_factory import get_db_connection

conn = get_db_connection()
cursor = conn.cursor()

# Find an infected devil
cursor.execute("""
    SELECT d.devil_id, d.name, s.name
    FROM devils d
    INNER JOIN sanctuaries s ON d.current_sanctuary_id = s.sanctuary_id
    INNER JOIN health_logs hl ON d.devil_id = hl.devil_id
    INNER JOIN strains st ON hl.detected_strain_id = st.strain_id
    WHERE d.status='Alive' AND st.strain_name IN ('DFT1', 'DFT2') AND hl.pcr_result='Positive'
    LIMIT 1
""")
row = cursor.fetchone()

if row:
    dev_id, dev_name, sanc_name = row
    # Get target clean sanctuary
    cursor.execute("SELECT sanctuary_id, name FROM sanctuaries WHERE is_clean_zone=True LIMIT 1")
    clean_sanc_id, clean_name = cursor.fetchone()
    
    print(f"Found positive animal: #{dev_id} ({dev_name}) at {sanc_name}.")
    print(f"Attempting illegal transfer to clean zone: {clean_name}...\n")
    
    try:
        cursor.execute("UPDATE devils SET current_sanctuary_id = %s WHERE devil_id = %s", (clean_sanc_id, dev_id))
        conn.commit()
        print("WARNING: Trigger did not block the transfer. Check if trigger was created.")
    except Exception as e:
        print("SUCCESS: Database blocked transfer natively!")
        print(f"Error thrown: {e}")
else:
    print("No infected devils found. Run Step 3 first.")
    
cursor.close()
conn.close()